# Neighborhood Filter

This guide showcases how to apply a neighborhood filter to unstructured grid data using UXarray's `neighborhood_filter` method.

A neighborhood filter replaces the value at each grid element with the result of a user-specified function (e.g. `np.mean`, `np.max`, `np.median`) applied to all grid elements whose centers fall within a circular neighborhood of radius `r` degrees around that element.

This is particularly useful for variable-resolution meshes, where a constant number of neighbors does not correspond to a constant spatial scale.

In [ ]:
from functools import partial

import numpy as np

import uxarray as ux

## Load Sample Data

We use the CSne30 cubed-sphere grid bundled with UXarray's test files.

In [ ]:
import pathlib

# Locate the bundled test meshfiles
repo_root = pathlib.Path(ux.__file__).parents[1]
grid_path = repo_root / "test" / "meshfiles" / "ugrid" / "outCSne30" / "outCSne30.ug"
data_path = repo_root / "test" / "meshfiles" / "ugrid" / "outCSne30" / "outCSne30_vortex.nc"

uxds = ux.open_dataset(str(grid_path), str(data_path))
uxda = uxds["psi"]
print(uxda)

## Basic Usage: Mean Filter

Apply a mean filter with a 5-degree radius. Each face value is replaced by the mean of all face values within 5° of that face's center.

In [ ]:
uxda_mean = uxda.neighborhood_filter(func=np.mean, r=5.0)
print(uxda_mean)

## Custom Functions via `functools.partial`

Any callable that accepts an `axis` keyword argument works as the filter function. Use `functools.partial` to pass additional arguments, for example to compute the 90th percentile in each neighborhood.

In [ ]:
# 90th-percentile filter
uxda_p90 = uxda.neighborhood_filter(func=partial(np.percentile, q=90), r=5.0)

# Maximum filter
uxda_max = uxda.neighborhood_filter(func=np.max, r=5.0)

print("p90 max:", uxda_p90.values.max(), "  filter max:", uxda_max.values.max())

## Dataset-Level Usage

`neighborhood_filter` is also available on `UxDataset`. It applies the filter to every data variable that is mapped to a grid element, leaving other variables (e.g. scalars with no grid dimension) unchanged.

In [ ]:
uxds_filtered = uxds.neighborhood_filter(func=np.mean, r=5.0)
print(uxds_filtered)

## Handling Extra Dimensions

If the data has extra leading dimensions (e.g. `time`), the filter is applied independently along the grid axis and all extra dimensions are preserved.

In [ ]:
from uxarray import UxDataArray

# Simulate two time steps
data = np.stack([uxda.values, uxda.values * 2.0])  # shape (2, n_face)
uxda_time = UxDataArray(
    data, dims=["time", "n_face"], uxgrid=uxda.uxgrid, name="psi_time"
)

filtered_time = uxda_time.neighborhood_filter(func=np.mean, r=5.0)
print("Input shape :", uxda_time.shape)
print("Output shape:", filtered_time.shape)
print("Output dims :", filtered_time.dims)

## Empty Neighborhoods

If the radius `r` is so small that no neighbor is found for a given element (unlikely for face-centered data because the query always finds the element itself, but possible for edge- or node-centered data with very small radii), the result for that element is `NaN` rather than an arbitrary value.